# 08 Second History snapshot (2026-10): label changes and reconciliation

The 2026-10 month files were cut to the slim columns locally (`python -m etl.history_slim`), loaded with `python -m etl.history bronze|silver --snapshot 2026-10`, and merged into `gold/fact_revert_label` as one commit (`python -m etl.gold_run --snapshot 2026-10 --steps ddl,dims,labels,label_change,recon,marts`). Checks: `python -m etl.snapshot_checks`.

In [1]:
import json, os, sys
from pathlib import Path
REPO = Path.cwd().resolve().parents[1] if Path.cwd().name == 'final' else Path.cwd()
sys.path.insert(0, str(REPO))
os.environ.setdefault('TRUTHLAG_HW2_DIR', str(REPO / 'ec-hw2'))
import pandas as pd
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 20)
from deltalake import DeltaTable
LAKE = REPO / 'data' / 'lake'
RUNS = LAKE / '_runs'
def run(name): return json.loads((RUNS / name).read_text())
def table(name, version=None, columns=None):
    return DeltaTable(str(LAKE / name), version=version).to_pandas(columns=columns)
def history(name, n=10):
    h = DeltaTable(str(LAKE / name)).history(n)
    return pd.DataFrame([{k: e.get(k) for k in ('version', 'operation', 'step', 'as_of', 'snapshot', 'rows')} for e in h])
sc = run('snapshot_checks.json')
print(sc['label_versions'], sc['passed'], 'passed,', sc['failed'], 'failed')
table('gold/dim_snapshot')

{'2026-09': 1, '2026-10': 2} 2 passed, 0 failed


,snapshot,snapshot_cutoff,loaded_at,revision_rows,snapshot_published_at
0,2026-10,2026-11-01 01:30:05+00:00,2026-11-15 10:42:43.488774+00:00,10625580,2026-11-04 21:28:39+00:00
1,2026-09,2026-10-01 01:38:27+00:00,2026-11-15 09:34:52.534288+00:00,5266843,NaT


## fact_revert_label history (one commit per snapshot load)

In [2]:
history('gold/fact_revert_label')

,version,operation,step,as_of,snapshot,rows
0,2,MERGE,None,None,None,None
1,1,MERGE,None,None,None,None
2,0,CREATE TABLE,None,None,None,None


## Checks

In [3]:
pd.DataFrame([{k: c[k] for k in ('check', 'ok')} | {'got': str(c['got'])[:160]} for c in sc['checks']])

,check,ok,got
0,"fact_revert_label versions (2026-09 load, 2026...",None,"[1, 2]"
1,history cohort at the 2026-09 load: reverted /...,True,"[19858, 19700, 17646]"
2,history cohort at the 2026-10 load: reverted /...,None,"[20925, 19695, 17641]"
3,change feed of the 2026-10 load: insert / upda...,None,"{'insert': 5358769, 'update_postimage': 40721,..."
4,update pre-images = post-images = mart_label_c...,True,"[40721, 40721]"
5,mart_label_change rows by class,None,"{'became_unreverted': 36, 'reverter_or_time_ch..."
6,"label changes among history-cohort revisions, ...",None,"{'became_unreverted': 5, 'reverter_or_time_cha..."
7,dim_snapshot,None,"[{'snapshot': '2026-09', 'snapshot_cutoff': '2..."
8,live: revisions with less than 4 days between ...,None,"[0, 279466]"
9,live_ext: revisions with less than 4 days betw...,None,"[1560837, 2397738]"


## Label changes between the snapshots (from the Change Data Feed)

In [4]:
lc = table('gold/mart_label_change')
lc.groupby(['from_snapshot', 'to_snapshot', 'change_class']).size()

from_snapshot  to_snapshot  change_class            
2026-09        2026-10      became_reverted             40683
                            became_unreverted              36
                            reverter_or_time_changed        2
dtype: int64

## Live cohort: stream label against the 2026-10 dump label

In [5]:
table('gold/mart_reconciliation').sort_values('recon_class')

,cohort_id,snapshot,label_version,recon_class,n_revisions,share
0,live,2026-10,2,both_reverted,17752,0.063521
2,live,2026-10,2,dump_only,325,0.001163
3,live,2026-10,2,neither,261323,0.935080
4,live,2026-10,2,not_in_dump,0,0.000000
1,live,2026-10,2,stream_only,66,0.000236


## E(L) of the live cohort: stream labels vs dump labels

In [6]:
el = table('gold/mart_el_curve')
el[(el.cohort_id == 'live') & (el.user_kind == 'all')].pivot_table(index=['wait_s', 'wait_label'], columns='label_source', values=['n_known', 'e_l']).round(4)

e_l          n_known         
label_source         dump  stream     dump   stream
wait_s wait_label                                  
0      0           1.0000  1.0000      0.0      0.0
60     1 min       0.7684  0.7763   4186.0   3986.0
300    5 min       0.6131  0.6095   6994.0   6958.0
900    15 min      0.5338  0.5287   8427.0   8398.0
3600   1 h         0.4266  0.4205  10365.0  10326.0
21600  6 h         0.2549  0.2477  13470.0  13405.0
86400  1 d         0.1133  0.1056  16028.0  15936.0
172800 2 d         0.0683  0.0628  16843.0  16699.0
345600 4 d         0.0000  0.0000  18077.0  17818.0